# Entraînement SH17 sur Kaggle (GPU) — projet détection d'EPI

**Avant de lancer :**
1. À droite, *Session options* → **Accelerator : GPU T4 x2** (ou P100) et **Internet : On**.
2. À droite, *Input* → **+ Add Input** → cherche **SH17** → ajoute le dataset « SH17 Dataset for PPE Detection ».
3. Le dépôt GitHub ghannemomar/mon-projet-deep-learning doit être **public**.
4. En haut à droite : **Save Version → Save & Run All (Commit)**. Le notebook tourne seul (3 à 4 h), même si tu fermes le navigateur.
5. À la fin : onglet **Output** de la version → télécharge **outputs.zip**.

In [ ]:
GITHUB_USER = "ghannemomar"   # <-- à modifier
REPO = "mon-projet-deep-learning"

## 1. Récupérer le code et installer les dépendances

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
!rm -rf {REPO}
!git clone https://github.com/{GITHUB_USER}/{REPO}.git
os.chdir(f"/kaggle/working/{REPO}")
!pip install -q -r requirements-train.txt
!git log --oneline | head -5

In [ ]:
import subprocess
def run(cmd, check=True):
    """Lance une commande, affiche sa sortie, et arrête le notebook si elle échoue."""
    print(">>>", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"Échec ({r.returncode}) : {cmd}")
    return r.returncode

In [ ]:
# vérification : le code d'entraînement est bien présent
run("ls src/models && python -c 'import src.models.train'")

## 2. Trouver SH17 et préparer les données (étape DVC `prepare` : split + redimensionnement)

In [ ]:
import glob, pathlib
hits = glob.glob("/kaggle/input/**/train_files.txt", recursive=True)
assert hits, "SH17 introuvable : ajoute le dataset dans Input (étape 2 en haut)"
RAW = str(pathlib.Path(hits[0]).parent)
print("Données brutes :", RAW, os.listdir(RAW))
assert os.path.isdir(f"{RAW}/images") and os.path.isdir(f"{RAW}/labels")
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/raw"):
    os.symlink(RAW, "data/raw")

In [ ]:
# dvc repro exécute le stage "prepare" défini dans dvc.yaml et écrit dvc.lock (empreintes des données)
ret = os.system("dvc repro prepare")
if ret != 0:
    print("dvc repro a échoué, lancement direct du même script")
    os.environ["SH17_RAW"] = RAW
    !python -m src.data.dataset
!cat data/processed/split_stats.json

## 3. MLflow (base SQLite + artefacts dans /mlflow, même chemin que dans docker-compose)

In [ ]:
os.makedirs("/mlflow/mlruns", exist_ok=True)
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:////mlflow/mlflow.db"
os.environ["MLFLOW_ARTIFACT_ROOT"] = "/mlflow/mlruns"

## 4. Entraînements

Run 1 = baseline YOLOv8n (même modèle que le benchmark publié : 58,0 mAP50).

In [ ]:
# récupère un éventuel correctif poussé pendant la préparation des données
run("git pull --ff-only", check=False)


In [ ]:
run("python -m src.models.train --model yolov8n --imgsz 640 --epochs 30 --name baseline-yolov8n-640")

Run 2 = architecture plus récente, même budget (YOLO26n, sinon YOLO11n si la version d'Ultralytics ne la connaît pas).

In [ ]:
if run("python -m src.models.train --model yolo26n --imgsz 640 --epochs 30 --name yolo26n-640", check=False) != 0:
    run("python -m src.models.train --model yolo11n --imgsz 640 --epochs 30 --name yolo11n-640")

## 5. Analyse (rappel par taille + tableau comparatif), export ONNX, Model Registry, paquet de sortie

In [ ]:
import shutil, json, random, glob
from PIL import Image

def package():
    run("python -m src.models.evaluate")
    run("python -m src.models.export")
    out = "/kaggle/working/outputs"
    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(f"{out}/examples", exist_ok=True)
    shutil.copytree("models", f"{out}/models")
    shutil.copytree("reports", f"{out}/reports")
    if os.path.exists("dvc.lock"):
        shutil.copy("dvc.lock", f"{out}/dvc.lock")
    shutil.copy("data/processed/split_stats.json", f"{out}/reports/split_stats.json")
    imgs = sorted(glob.glob("data/processed/images/test/*.jpg"))
    random.Random(0).shuffle(imgs)
    names = []
    for p in imgs[:6]:  # 6 images du test réduites à 800 px pour la galerie du frontend
        im = Image.open(p); im.thumbnail((800, 800)); n = os.path.basename(p)
        im.save(f"{out}/examples/{n}", quality=85); names.append(n)
    json.dump(names, open(f"{out}/examples/index.json", "w"))
    shutil.make_archive(f"{out}/mlflow", "zip", "/mlflow")
    if os.path.exists("/kaggle/working/outputs.zip"):
        os.remove("/kaggle/working/outputs.zip")
    shutil.make_archive("/kaggle/working/outputs", "zip", out)
    print(open("reports/comparison.csv").read())
    print(open("reports/recall_by_size.csv").read())
    print(open("models/metadata.json").read())

package()

## 6. Run 3 : effet de la résolution (35,7 % de petits objets) — baseline à 960 px

Si ce run échoue ou manque de temps, le paquet de l'étape 5 est déjà prêt.

In [ ]:
try:
    run("python -m src.models.train --model yolov8n --imgsz 960 --epochs 20 --batch 16 --name yolov8n-960")
    package()   # refait l'analyse : le meilleur run (sur la validation) est ré-exporté
except Exception as e:
    print("Run 3 ignoré :", e)